In [1]:
import sys
sys.executable

'/home/nicolofiaba/Desktop/jupyter-notebooks/Projects/proj3-pytorch-classifier/.venv/bin/python'

# PyTorch Classifier - Wine Quality Detection

In [46]:
import numpy as np
import torch
from torchvision.transforms import v2
import kagglehub
from pathlib import Path
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
class WineDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X = self.data[idx]
        y = self.labels[idx] - 3

        return X, y

# Neural Network Approach (PyTorch)

## Dataset and Dataloader preparation

In [48]:
data = pd.read_csv("data/winequality-red.csv")
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

In [49]:
from sklearn.model_selection import train_test_split

scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [50]:
scaler.fit(X_train)
X_train_sc = scaler.transform(X_train)
X_test_sc = scaler.transform(X_test)

In [51]:
wd_train = WineDataset(X_train_sc, y_train)
wd_test = WineDataset(X_test_sc, y_test)

In [52]:
train_dataloader = DataLoader(wd_train, batch_size=64, shuffle=True)
test_dataloader = DataLoader(wd_test, batch_size=64, shuffle=False)

## Model definition and Training

In [53]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [54]:
from torch import nn

In [55]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(11, 33),
            nn.ReLU(),
            nn.Linear(33, 33),
            nn.ReLU(),
            nn.Linear(33, 6)
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

In [66]:
model = NeuralNetwork().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [84]:
from sklearn.metrics import confusion_matrix

In [73]:
epochs = 20

for epoch in range(epochs):

    # Start the training process
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for X, y in train_dataloader: 
    
        pred = model(X)
        loss = loss_fn(pred, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        predicted = pred.argmax(dim=1)
        train_correct += (predicted == y).sum().item()
        train_total += y.size(0)
        
    train_loss /= len(train_dataloader)
    train_accuracy = train_correct / train_total

    # Validation
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for X, y in test_dataloader:
            pred = model(X)
            loss = loss_fn(pred, y)

            val_loss += loss.item()

            predicted = pred.argmax(dim=1)
            val_correct += (predicted == y).sum().item()
            val_total += y.size(0)

    val_loss /= len(test_dataloader)
    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Train loss: {train_loss:.4f} "
        f"| Train acc: {train_accuracy:.3f} "
        f"| Val loss: {val_loss:.4f} "
        f"| Val acc: {val_accuracy:.3f}"
    )

Epoch 1/20 | Train loss: 0.6579 | Train acc: 0.710 | Val loss: 1.0488 | Val acc: 0.597
Epoch 2/20 | Train loss: 0.6606 | Train acc: 0.711 | Val loss: 1.0538 | Val acc: 0.609
Epoch 3/20 | Train loss: 0.6570 | Train acc: 0.722 | Val loss: 1.0569 | Val acc: 0.603
Epoch 4/20 | Train loss: 0.6544 | Train acc: 0.722 | Val loss: 1.0567 | Val acc: 0.584
Epoch 5/20 | Train loss: 0.6539 | Train acc: 0.720 | Val loss: 1.0502 | Val acc: 0.603
Epoch 6/20 | Train loss: 0.6552 | Train acc: 0.717 | Val loss: 1.0573 | Val acc: 0.603
Epoch 7/20 | Train loss: 0.6504 | Train acc: 0.717 | Val loss: 1.0627 | Val acc: 0.600
Epoch 8/20 | Train loss: 0.6497 | Train acc: 0.719 | Val loss: 1.0609 | Val acc: 0.600
Epoch 9/20 | Train loss: 0.6502 | Train acc: 0.722 | Val loss: 1.0643 | Val acc: 0.609
Epoch 10/20 | Train loss: 0.6481 | Train acc: 0.720 | Val loss: 1.0638 | Val acc: 0.591
Epoch 11/20 | Train loss: 0.6450 | Train acc: 0.720 | Val loss: 1.0681 | Val acc: 0.581
Epoch 12/20 | Train loss: 0.6458 | Train 

In [85]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for X, y in test_dataloader:
        pred = model(X)

        y_true.extend(y.numpy())
        y_pred.extend(pred.argmax(dim=1).numpy())

cm = confusion_matrix(y_true, y_pred)

print(cm)

[[  0   1   2   0   0   0]
 [  0   0   6   1   0   0]
 [  0   0 108  32   1   0]
 [  0   0  42  69   9   1]
 [  0   0   0  28  15   1]
 [  0   0   0   1   3   0]]
